# PC3 Dynamics — The Third Dimension Across Sites and Scales

Models tend to organize class centroids into a 2D ring (PC1+PC2 ≈ 1.0).
But PC3 is not static — it expands and contracts with training.

**Hypothesis**: PC3 is operational. The model uses it as working space:
- MLP uses PC3 early (rising first) as it begins to organize
- Attention compresses PC3 to minimum just before grokking (ring at maximum planarity)
- Post-grokking: PC3 re-expands in the attention stream

**Two scales**:
1. **Activation-space PC3** (from `repr_geometry`) — how class centroids spread into 3D at each site
2. **Weight-space PC3** (from `parameter_snapshot` + global W_in PCA) — how frequency group centroids move in 3D neuron space

**Question**: Are these in sync? Does one scale lead the other?

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader
from miscope.analysis.library import extract_neuron_weight_matrix

RESULTS_BASE = '../results/modulo_addition_1layer'
FAMILY_NAME  = 'modulo_addition_1layer'
family = load_family(FAMILY_NAME)

# Health spectrum: fast healthy → canon healthy → late → degraded
VARIANTS = [
    {'label': 'p109/s485/ds598', 'dir': 'modulo_addition_1layer_p109_seed485_dseed598', 'grok': 5243,  'health': 'healthy', 'color': '#1f77b4', 'prime': 109, 'seed': 485, 'dseed': 598},
    {'label': 'p113/s999/ds598', 'dir': 'modulo_addition_1layer_p113_seed999_dseed598', 'grok': 12448, 'health': 'healthy', 'color': '#2ca02c', 'prime': 113, 'seed': 999, 'dseed': 598},
    {'label': 'p101/s485/ds598', 'dir': 'modulo_addition_1layer_p101_seed485_dseed598', 'grok': 16656, 'health': 'late',    'color': '#ff7f0e', 'prime': 101, 'seed': 485, 'dseed': 598},
    {'label': 'p101/s999/ds598', 'dir': 'modulo_addition_1layer_p101_seed999_dseed598', 'grok': -1,    'health': 'late',    'color': '#d62728', 'prime': 101, 'seed': 999, 'dseed': 598},
    {'label': 'p59/s485/ds598',  'dir': 'modulo_addition_1layer_p59_seed485_dseed598',  'grok': -1,    'health': 'degraded','color': '#9467bd', 'prime': 59,  'seed': 485, 'dseed': 598},
]

SITES = ['attn_out', 'mlp_out', 'resid_post']
SITE_COLORS = {'attn_out': '#2ca02c', 'mlp_out': '#d62728', 'resid_post': '#9467bd'}
SITE_LABELS = {'attn_out': 'Attn (mid-stream)', 'mlp_out': 'MLP (output delta)', 'resid_post': 'Resid Post'}

def load_repr_geometry(variant_dir):
    loader = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')
    s = loader.load_summary('repr_geometry')
    return s

repr_data = {v['label']: load_repr_geometry(v['dir']) for v in VARIANTS}
print('Loaded repr_geometry summaries.')

## Cell 1: PC3 Timeseries — All Sites, Single Variant

PC3 variance fraction at each activation site over training.
Lower = centroids compressed into the PC1-PC2 ring plane.
Higher = centroids have spread into a third dimension.

Expected signature for healthy grokkers:
- MLP PC3 peaks BEFORE grokking (MLP uses 3D first)
- Attn PC3 compresses to minimum AT grokking
- Both expand post-grokking

In [ ]:
FOCUS_VARIANT = 'p113/s999/ds598'
FOCUS_GROK    = next(v['grok'] for v in VARIANTS if v['label'] == FOCUS_VARIANT)

s = repr_data[FOCUS_VARIANT]
epochs = s['epochs']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=['PC3 variance fraction (centroid 3D budget)',
                                    'PC1 + PC2 variance fraction (ring-plane budget)'])

for site in SITES:
    pc3 = s[f'{site}_pca_var_pc3']
    pc12 = s[f'{site}_pca_var_pc1'] + s[f'{site}_pca_var_pc2']
    color = SITE_COLORS[site]
    label = SITE_LABELS[site]

    # Mark minimum PC3
    min_idx = np.argmin(pc3)

    fig.add_trace(go.Scatter(x=epochs, y=pc3, mode='lines', name=label,
                             legendgroup=site, showlegend=True,
                             line=dict(color=color, width=2.5),
                             hovertemplate=f'{label}<br>Epoch %{{x}}<br>PC3: %{{y:.3f}}<extra></extra>'),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=[epochs[min_idx]], y=[pc3[min_idx]], mode='markers',
                             marker=dict(color=color, size=9, symbol='x'),
                             showlegend=False, legendgroup=site,
                             hovertemplate=f'{label} min<br>ep=%{{x}}<br>%{{y:.3f}}<extra></extra>'),
                  row=1, col=1)

    fig.add_trace(go.Scatter(x=epochs, y=pc12, mode='lines', name=label,
                             legendgroup=site, showlegend=False,
                             line=dict(color=color, width=2.5)),
                  row=2, col=1)

if FOCUS_GROK > 0:
    for row in [1, 2]:
        fig.add_vline(x=FOCUS_GROK, line=dict(color='black', dash='dash', width=1.5), row=row, col=1)

fig.update_yaxes(title_text='PC3 variance fraction', row=1, col=1)
fig.update_yaxes(title_text='PC1+PC2 variance fraction', row=2, col=1)
fig.update_xaxes(title_text='Epoch', row=2, col=1)
fig.update_layout(
    title=f'{FOCUS_VARIANT} — PC3 dynamics (× = PC3 minimum; dashed = grokking)',
    height=650, legend=dict(orientation='h', y=1.03),
)
fig.show()

## Cell 2: Cross-Site PC3 Overlay — All Variants

One row per variant. All three sites on the same axes to see synchrony.
X marker = PC3 minimum per site. Vertical dashes = grokking epoch.

In [ ]:
n = len(VARIANTS)
fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=[f"{v['label']} ({v['health']}, grok={v['grok']})" for v in VARIANTS])

pc3_events = []  # (label, site, min_epoch, min_val, grok, lead)

for row_idx, v in enumerate(VARIANTS, 1):
    s   = repr_data[v['label']]
    epochs = s['epochs']

    for site in SITES:
        pc3   = s[f'{site}_pca_var_pc3']
        color = SITE_COLORS[site]
        label = SITE_LABELS[site]

        min_idx   = np.argmin(pc3)
        min_epoch = int(epochs[min_idx])
        min_val   = float(pc3[min_idx])
        lead      = (v['grok'] - min_epoch) if v['grok'] > 0 else None
        pc3_events.append({'variant': v['label'], 'site': site, 'health': v['health'],
                            'min_epoch': min_epoch, 'min_val': min_val,
                            'grok': v['grok'], 'lead': lead})

        fig.add_trace(go.Scatter(x=epochs, y=pc3, mode='lines',
                                 name=label, legendgroup=site,
                                 showlegend=(row_idx == 1),
                                 line=dict(color=color, width=2),
                                 hovertemplate=f'{label}<br>Epoch %{{x}}<br>PC3: %{{y:.3f}}<extra></extra>'),
                      row=row_idx, col=1)

        fig.add_trace(go.Scatter(x=[min_epoch], y=[min_val], mode='markers',
                                 marker=dict(color=color, size=9, symbol='x'),
                                 showlegend=False, legendgroup=site),
                      row=row_idx, col=1)

    if v['grok'] > 0:
        fig.add_vline(x=v['grok'], line=dict(color='black', dash='dash', width=1.5),
                      row=row_idx, col=1)

    fig.update_yaxes(title_text='PC3 frac', row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=n, col=1)
fig.update_layout(title='PC3 dynamics — health spectrum (× = site minimum; dashed = grokking)',
                  height=220 * n, legend=dict(orientation='h', y=1.02))
fig.show()

# Summary table
print(f'\n{"Variant":<25} {"Site":<18} {"Health":<12} {"MinEp":<8} {"MinVal":<8} {"Lead":<8}')
print('-' * 80)
for e in pc3_events:
    lead_str = f'{e["lead"]:+d}' if e['lead'] is not None else 'N/A'
    print(f"{e['variant']:<25} {e['site']:<18} {e['health']:<12} {e['min_epoch']:<8} {e['min_val']:<8.3f} {lead_str}")

## Cell 3: PC3 Minimum Timing — Lead/Lag Relative to Grokking

For each grokked variant, where does each site's PC3 minimum fall relative to grokking?
Positive lead = PC3 compresses before grokking; negative = after.

Hypothesis:
- Attn PC3 minimum closely precedes grokking (maximum ring planarity just before transition)
- MLP PC3 minimum precedes attn (MLP organizes first, releases 3D space)

In [ ]:
grokked_events = [e for e in pc3_events if e['lead'] is not None]

fig = go.Figure()

for site in SITES:
    site_events = [e for e in grokked_events if e['site'] == site]
    variant_labels = [e['variant'] for e in site_events]
    leads = [e['lead'] for e in site_events]

    fig.add_trace(go.Scatter(
        x=variant_labels, y=leads,
        mode='markers+lines',
        name=SITE_LABELS[site],
        marker=dict(color=SITE_COLORS[site], size=12),
        line=dict(color=SITE_COLORS[site], width=1.5, dash='dot'),
        hovertemplate=f'{SITE_LABELS[site]}<br>%{{x}}<br>Lead: %{{y:+d}} epochs<extra></extra>',
    ))

fig.add_hline(y=0, line=dict(color='black', dash='dot', width=1))
fig.update_layout(
    title='PC3 minimum lead before grokking per site<br>Positive = PC3 compresses before grokking',
    xaxis_title='Variant',
    yaxis_title='Lead (epochs) = grokking_epoch − min_pc3_epoch',
    height=420,
)
fig.show()

## Cell 4: mean_dim vs PC3 — Two Dimensionality Signals

Two different views of dimensionality:
- **PC3 variance fraction** — how class *centroids* spread into a 3rd direction (inter-class geometry)
- **mean_dim** — average effective dimensionality of each *class cloud* (intra-class spread)

These are complementary: PC3 captures global structure (is the ring 3D?); mean_dim captures
local cloud compactness (are each class's activations tight?).

Expected: mean_dim should crash (classes collapse to points) while PC3 may expand (the ring
becomes 3D). They measure orthogonal aspects of geometric maturation.

In [ ]:
FOCUS_VARIANT = 'p113/s999/ds598'
FOCUS_GROK    = next(v['grok'] for v in VARIANTS if v['label'] == FOCUS_VARIANT)

s = repr_data[FOCUS_VARIANT]
epochs = s['epochs']

fig = make_subplots(rows=len(SITES), cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=[SITE_LABELS[site] for site in SITES])

for row_idx, site in enumerate(SITES, 1):
    pc3     = s[f'{site}_pca_var_pc3']
    md      = s[f'{site}_mean_dim']
    color   = SITE_COLORS[site]

    # PC3 on primary y-axis
    fig.add_trace(go.Scatter(x=epochs, y=pc3, mode='lines', name='PC3 frac',
                             legendgroup='pc3', showlegend=(row_idx == 1),
                             line=dict(color=color, width=2.5)),
                  row=row_idx, col=1)

    # mean_dim normalised to [0,1] for overlay
    md_norm = (md - md.min()) / (md.max() - md.min() + 1e-9)
    fig.add_trace(go.Scatter(x=epochs, y=md_norm, mode='lines',
                             name='mean_dim (norm)', legendgroup='mean_dim',
                             showlegend=(row_idx == 1),
                             line=dict(color='#aaaaaa', width=1.5, dash='dot')),
                  row=row_idx, col=1)

    if FOCUS_GROK > 0:
        fig.add_vline(x=FOCUS_GROK, line=dict(color='black', dash='dash', width=1.5),
                      row=row_idx, col=1)

    fig.update_yaxes(title_text='PC3 / norm dim', row=row_idx, col=1)

fig.update_xaxes(title_text='Epoch', row=len(SITES), col=1)
fig.update_layout(
    title=f'{FOCUS_VARIANT} — PC3 (solid) vs mean_dim normalised (dotted gray)\nDashed = grokking',
    height=600, legend=dict(orientation='h', y=1.03),
)
fig.show()

## Cell 5: Weight-Space Group Centroid Trajectories

In the macro surface view (mlp_macro_surface.ipynb), we see MLP neurons as a 3D cloud
in global W_in PCA space. Here we collapse each frequency group to its centroid and
track that centroid's trajectory over training.

**What this shows**: how each frequency group's collective position in weight space evolves.
PC3 of each group centroid = how far the group has moved 'out of the ring plane' in weight space.

**Link to activation space**: if weight-space PC3 and activation-space PC3 are synchronized,
the geometry is coherent across scales.

In [ ]:
GROUP_COLORS = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#17becf', '#bcbd22', '#7f7f7f',
]

def load_group_centroid_trajectories(variant_dir, variant_obj) -> dict:
    """Compute per-group centroid trajectory in global W_in PCA space.

    Returns:
        epochs:      (n_epochs,)
        centroids:   (n_epochs, n_groups, 3) — group centroids in global PCA space
        group_freqs: (n_groups,) 1-indexed
        var_ratio:   (3,) PCA explained variance
    """
    loader = ArtifactLoader(f'{RESULTS_BASE}/{variant_dir}/artifacts')

    ngpca       = loader.load_cross_epoch('neuron_group_pca')
    group_idx   = ngpca['neuron_group_idx']              # (d_mlp,)
    group_freqs = (ngpca['group_freqs'] + 1).astype(int) # 1-indexed
    epochs_arr  = ngpca['epochs']                        # (n_epochs,)

    # Fit global PCA on final epoch W_in
    epoch_list = sorted(epochs_arr.tolist())
    final_snap = loader.load_epoch('parameter_snapshot', epoch_list[-1])
    W_in_final = extract_neuron_weight_matrix(final_snap)  # (d_model, d_mlp)

    pca = PCA(n_components=3, random_state=0)
    pca.fit(W_in_final.T)  # (d_mlp, d_model) — neurons as samples

    n_groups = len(group_freqs)
    all_centroids = np.zeros((len(epoch_list), n_groups, 3), dtype=np.float32)

    for ep_i, epoch in enumerate(epoch_list):
        snap  = loader.load_epoch('parameter_snapshot', epoch)
        W_in  = extract_neuron_weight_matrix(snap)         # (d_model, d_mlp)
        proj  = pca.transform(W_in.T)                      # (d_mlp, 3)
        for g in range(n_groups):
            mask = group_idx == g
            if mask.any():
                all_centroids[ep_i, g] = proj[mask].mean(axis=0)

    return {
        'epochs':      np.array(epoch_list),
        'centroids':   all_centroids,     # (n_epochs, n_groups, 3)
        'group_freqs': group_freqs,
        'var_ratio':   pca.explained_variance_ratio_,
    }


# Load for canon model
WEIGHT_FOCUS = 'p113/s999/ds598'
WEIGHT_FOCUS_GROK = next(v['grok'] for v in VARIANTS if v['label'] == WEIGHT_FOCUS)
WEIGHT_FOCUS_OBJ  = next(v for v in VARIANTS if v['label'] == WEIGHT_FOCUS)

print(f'Loading weight-space group centroid trajectories for {WEIGHT_FOCUS}...')
weight_traj = load_group_centroid_trajectories(WEIGHT_FOCUS_OBJ['dir'],
                                               family.get_variant(prime=WEIGHT_FOCUS_OBJ['prime'],
                                                                  seed=WEIGHT_FOCUS_OBJ['seed'],
                                                                  data_seed=WEIGHT_FOCUS_OBJ['dseed']))
print(f'  {len(weight_traj["epochs"])} epochs, {len(weight_traj["group_freqs"])} groups: {weight_traj["group_freqs"].tolist()}')
print(f'  Global W_in PCA variance: {weight_traj["var_ratio"]*100}')

In [ ]:
# Plot PC1, PC2, PC3 of each group centroid over training
epochs_w    = weight_traj['epochs']
centroids_w = weight_traj['centroids']   # (n_epochs, n_groups, 3)
group_freqs = weight_traj['group_freqs']
n_groups    = len(group_freqs)

pc_labels = ['PC1 (weight space)', 'PC2 (weight space)', 'PC3 (weight space)']
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=pc_labels)

for g in range(n_groups):
    color = GROUP_COLORS[g % len(GROUP_COLORS)]
    freq  = int(group_freqs[g])
    for pc_dim in range(3):
        fig.add_trace(go.Scatter(
            x=epochs_w,
            y=centroids_w[:, g, pc_dim],
            mode='lines',
            name=f'freq {freq}',
            legendgroup=f'freq{freq}',
            showlegend=(pc_dim == 0),
            line=dict(color=color, width=2),
            hovertemplate=f'freq {freq}<br>Epoch %{{x}}<br>PC{pc_dim+1}: %{{y:.3f}}<extra></extra>',
        ), row=pc_dim + 1, col=1)

if WEIGHT_FOCUS_GROK > 0:
    for row in [1, 2, 3]:
        fig.add_vline(x=WEIGHT_FOCUS_GROK, line=dict(color='black', dash='dash', width=1.5),
                      row=row, col=1)

for row, label in enumerate(['PC1', 'PC2', 'PC3'], 1):
    fig.update_yaxes(title_text=label, row=row, col=1)
fig.update_xaxes(title_text='Epoch', row=3, col=1)

fig.update_layout(
    title=f'{WEIGHT_FOCUS} — frequency group centroid trajectories in global W_in PCA space\n'
          f'(var: PC1={weight_traj["var_ratio"][0]:.1%} PC2={weight_traj["var_ratio"][1]:.1%} '
          f'PC3={weight_traj["var_ratio"][2]:.1%}) | dashed = grokking',
    height=700, legend=dict(orientation='h', y=1.03),
)
fig.show()

## Cell 6: Cross-Scale Comparison — Weight-Space PC3 vs Activation-Space PC3

The key synchrony question: does PC3 in weight space (group centroid height) move
in sync with PC3 in activation space (class centroid ring-plane budget)?

Weight-space PC3 = absolute coordinate of group centroid in global W_in PCA.
For comparison: use the abs-deviation from the mean PC3 to capture 'expansion' regardless of sign.

Activation-space PC3 = variance fraction of class centroids in 3rd principal direction.

In [ ]:
s_act = repr_data[WEIGHT_FOCUS]
epochs_act = s_act['epochs']

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=[
        'Activation-space PC3 (class centroid variance fraction) per site',
        'Weight-space |PC3| (group centroid deviation from mean) per frequency group',
    ],
)

# Row 1: activation-space PC3
for site in SITES:
    pc3 = s_act[f'{site}_pca_var_pc3']
    fig.add_trace(go.Scatter(
        x=epochs_act, y=pc3,
        mode='lines', name=SITE_LABELS[site],
        legendgroup=f'act_{site}',
        line=dict(color=SITE_COLORS[site], width=2.5),
    ), row=1, col=1)

# Row 2: weight-space PC3 deviation
pc3_vals_w = centroids_w[:, :, 2]  # (n_epochs, n_groups)
mean_pc3   = pc3_vals_w.mean(axis=1, keepdims=True)
pc3_dev    = np.abs(pc3_vals_w - mean_pc3)  # deviation from global mean

for g in range(n_groups):
    color = GROUP_COLORS[g % len(GROUP_COLORS)]
    freq  = int(group_freqs[g])
    fig.add_trace(go.Scatter(
        x=epochs_w, y=pc3_dev[:, g],
        mode='lines', name=f'freq {freq} (W_in)',
        legendgroup=f'w_{g}',
        line=dict(color=color, width=2, dash='dot'),
    ), row=2, col=1)

if WEIGHT_FOCUS_GROK > 0:
    for row in [1, 2]:
        fig.add_vline(x=WEIGHT_FOCUS_GROK, line=dict(color='black', dash='dash', width=1.5),
                      row=row, col=1)

fig.update_yaxes(title_text='PC3 variance fraction', row=1, col=1)
fig.update_yaxes(title_text='|PC3 deviation| (a.u.)', row=2, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=2, col=1)
fig.update_layout(
    title=f'{WEIGHT_FOCUS} — activation-space vs weight-space PC3 dynamics | dashed = grokking',
    height=700, legend=dict(orientation='h', y=1.03),
)
fig.show()

## Cell 7: 3D Group Centroid Trajectories (Spatial View)

Show each frequency group's centroid path in 3D weight space over training.
This makes the 3rd dimension visible as a spatial trajectory, not just a timeseries.

Color encodes training phase (early = light, late = dark) or epoch index.

In [ ]:
n_epochs_w = len(epochs_w)
phase = np.linspace(0, 1, n_epochs_w)

fig = go.Figure()

for g in range(n_groups):
    color = GROUP_COLORS[g % len(GROUP_COLORS)]
    freq  = int(group_freqs[g])
    traj  = centroids_w[:, g, :]  # (n_epochs, 3)

    # Full trajectory line
    fig.add_trace(go.Scatter3d(
        x=traj[:, 0], y=traj[:, 1], z=traj[:, 2],
        mode='lines',
        name=f'freq {freq}',
        line=dict(color=color, width=3),
        hovertemplate=f'freq {freq}<br>PC1=%{{x:.2f}}<br>PC2=%{{y:.2f}}<br>PC3=%{{z:.2f}}<extra></extra>',
    ))

    # Start and end markers
    fig.add_trace(go.Scatter3d(
        x=[traj[0, 0]], y=[traj[0, 1]], z=[traj[0, 2]],
        mode='markers',
        marker=dict(color=color, size=5, symbol='circle-open'),
        showlegend=False, name=f'freq {freq} start',
    ))
    fig.add_trace(go.Scatter3d(
        x=[traj[-1, 0]], y=[traj[-1, 1]], z=[traj[-1, 2]],
        mode='markers',
        marker=dict(color=color, size=6, symbol='circle'),
        showlegend=False, name=f'freq {freq} end',
    ))

vr = weight_traj['var_ratio']
fig.update_layout(
    title=f'{WEIGHT_FOCUS} — frequency group centroid paths in W_in PCA space<br>'
          f'PC1={vr[0]:.1%} PC2={vr[1]:.1%} PC3={vr[2]:.1%} | open circle = epoch 0, filled = final',
    scene=dict(
        xaxis_title='PC1', yaxis_title='PC2', zaxis_title='PC3',
        aspectmode='cube',
    ),
    height=680, width=820,
    legend=dict(x=1.0, y=0.9),
)
fig.show()

## Cell 8: Cross-Variant Weight-Space Group Centroid PC3

Same group-centroid PC3 timeseries for all reference variants.
Does the 3D expansion/contraction pattern in weight space match health?
Does degraded vs healthy show qualitatively different trajectories?

In [ ]:
# Load all variants (takes a while)
all_weight_traj = {}
for v in VARIANTS:
    print(f'Loading {v["label"]}...')
    try:
        variant_obj = family.get_variant(prime=v['prime'], seed=v['seed'], data_seed=v['dseed'])
        all_weight_traj[v['label']] = load_group_centroid_trajectories(v['dir'], variant_obj)
        print(f'  OK — {len(all_weight_traj[v["label"]]["group_freqs"])} groups')
    except Exception as ex:
        print(f'  FAILED: {ex}')

In [ ]:
# Cross-variant: mean |PC3| across all groups as a summary signal
fig = go.Figure()

for v in VARIANTS:
    if v['label'] not in all_weight_traj:
        continue
    wt = all_weight_traj[v['label']]
    pc3_coords  = wt['centroids'][:, :, 2]        # (n_epochs, n_groups)
    mean_abs_pc3 = np.abs(pc3_coords - pc3_coords.mean(axis=1, keepdims=True)).mean(axis=1)

    fig.add_trace(go.Scatter(
        x=wt['epochs'], y=mean_abs_pc3,
        mode='lines',
        name=f"{v['label']} ({v['health']})",
        line=dict(color=v['color'], width=2),
        hovertemplate=f"{v['label']}<br>Epoch %{{x}}<br>Mean |PC3|: %{{y:.3f}}<extra></extra>",
    ))

    if v['grok'] > 0:
        fig.add_vline(x=v['grok'], line=dict(color=v['color'], dash='dash', width=1.5))

fig.update_layout(
    title='Mean |PC3 deviation| of frequency group centroids in W_in weight space<br>'
          'Dashed = grokking epoch | higher = groups more spread in 3rd dimension',
    xaxis_title='Epoch',
    yaxis_title='Mean |PC3 deviation|',
    height=450,
)
fig.show()